In [1]:
suppressPackageStartupMessages({
  library(data.table)  # fread, rbindlist
  library(dplyr)       # filter, mutate, arrange, distinct
  library(tidyr)       # pivot_wider
  library(stringr)     # str_to_title, gsub wrappers
  library(ggplot2)     # plotting
  library(ggrepel)     # geom_text_repel
  library(grid)        # unit()
  library(DESeq2)      # DE analysis
  library(qvalue)      # qvalue FDR
  library(parallel)    # mclapply
  library(tidyverse)
})

In [3]:
source('../../00-utilities/functions/r/base-deseq2.R')

In [4]:
metadata = read.csv("../../../data/rna/pseudobulk/outputs/pbmc_sample_kit_metadata.csv")

In [5]:
# Define comparisons and gene list files
comparisons <- list(
  # Treatment progression comparisons
  list(tp1 = "PreTx",   tp2 = "EI",      gene_file = "pbmc_l3_PreTx-vs-EI_filtered_gene_list.csv"),
  list(tp1 = "PreTx",   tp2 = "PI2C",    gene_file = "pbmc_l3_PreTx-vs-PI2C_filtered_gene_list.csv"),
  list(tp1 = "PI2C",    tp2 = "EI",      gene_file = "pbmc_l3_PI2C-vs-EI_filtered_gene_list.csv"),
  
  # Post-transplant progression comparisons
  list(tp1 = "EI",      tp2 = "ASCT60d", gene_file = "pbmc_l3_EI-vs-ASCT60d_filtered_gene_list.csv"),
  list(tp1 = "EI",      tp2 = "ASCT1y",  gene_file = "pbmc_l3_EI-vs-ASCT1y_filtered_gene_list.csv"),
  list(tp1 = "EI",      tp2 = "ASCT2y",  gene_file = "pbmc_l3_EI-vs-ASCT2y_filtered_gene_list.csv"),
  list(tp1 = "ASCT60d", tp2 = "ASCT1y",  gene_file = "pbmc_l3_ASCT60d-vs-ASCT1y_filtered_gene_list.csv"),
  list(tp1 = "ASCT1y",  tp2 = "ASCT2y",  gene_file = "pbmc_l3_ASCT1y-vs-ASCT2y_filtered_gene_list.csv")
)

In [6]:
# Loop over each comparison
for (comp in comparisons) {
  timepoint_1 <- comp$tp1
  timepoint_2 <- comp$tp2
  gene_file <- comp$gene_file

  # Define output file using timepoints
  out_file <- paste0("deseq2_results_pbmc_", timepoint_1, "_vs_", timepoint_2, ".csv")

  # Filter metadata for only the two timepoints
  metadata_sub <- metadata %>%
    filter(
      manual.category == "tumor_pbmc",
      label.visitDetails %in% c(timepoint_1, timepoint_2)
    )

  # List of pseudobulk expression files
  aggregated_count_file_list <- paste0(
    "../../../data/rna/pseudobulk/outputs/pbmc_l3_raw_gexp_celltypes_per_samplekit/",
    unique(metadata_sub$sample.sampleKitGuid), ".csv"
  )
  df_list <- read_pseudobulk_expression(aggregated_count_file_list)

  # Drop failing samples for minimum cell count (10) and minimum total counts (1000) thresholds
  dropped <- subset(metadata_sub, keep == "False")
  drop_ids <- paste(dropped$sample.sampleKitGuid, dropped$aifi_plot_l3, sep = ":")
  df_list <- lapply(df_list, function(df) {
    df[, !(names(df) %in% drop_ids), drop = FALSE]
  })

  # Coverage check to remove cell types where there's only 1 sample or an absence of a necessary condition
  per_df_ct <- lapply(df_list, function(df) {
    # Skip dataframes with no columns (after filtering)
    if (ncol(df) == 0) return(data.frame(kit_id = character(0), celltype = character(0)))
    
    kit_id <- sub(":.*", "", names(df)[1])
    data.frame(
      kit_id   = kit_id,
      celltype = sub("^[^:]+:", "", names(df))
    )
  })
  all_ct <- do.call(rbind, per_df_ct)
  kit_info <- metadata_sub %>% 
    distinct(sample.sampleKitGuid, label.visitDetails, subject.biologicalSex)

  all_ct_with_meta <- merge(all_ct, kit_info,
    by.x = "kit_id", by.y = "sample.sampleKitGuid"
  )

  ct_coverage_by <- all_ct_with_meta %>%
    distinct(kit_id, celltype, label.visitDetails, subject.biologicalSex) %>%
    group_by(celltype, label.visitDetails, subject.biologicalSex) %>%
    summarise(n_kits = n(), .groups = "drop")

  # Plot the cell type coverage across sex & time points
  plot_dir <- "../../../data/rna/pseudobulk/results/ct_coverage_plots"
  dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

  p <- ggplot(ct_coverage_by, aes(x = n_kits, fill = label.visitDetails)) +
    geom_histogram(binwidth = 1, boundary = 0, color = "white", position = "dodge") +
    facet_wrap(~subject.biologicalSex, nrow = 1) +
    labs(
      title = paste0("Cell-type coverage: ", timepoint_1, " vs ", timepoint_2),
      x = "# of kits containing the cell type",
      y = "Number of cell types",
      fill = "Visit Details"
    ) +
    theme_bw() +
    theme(legend.position = "right")

  outfile_png <- file.path(
    plot_dir,
    paste0("ct_coverage_pbmc_", timepoint_1, "_vs_", timepoint_2, ".png")
  )
  ggsave(outfile_png, p, width = 7, height = 3, dpi = 300, bg = "white")

  # Identify problematic cell types
  n_of_one_cells <- unique(ct_coverage_by[ct_coverage_by$n_kits <= 1, ]$celltype)

  all_combos <- expand_grid(
    celltype = unique(ct_coverage_by$celltype),
    label.visitDetails = unique(ct_coverage_by$label.visitDetails),
    subject.biologicalSex = unique(ct_coverage_by$subject.biologicalSex)
  )

  # Drop any missing combinations
  missing_conditions <- all_combos %>%
    anti_join(ct_coverage_by, by = c("celltype", "label.visitDetails", "subject.biologicalSex")) %>%
    pull(celltype) %>%
    unique()

  # Get unique cell types for analysis
  celltypes <- unique(sub(".*:", "", unlist(lapply(df_list, names))))
  celltypes <- setdiff(celltypes, c(n_of_one_cells, missing_conditions))

  # Load condition-specific gene list
  filtered_gene_set <- read.csv(file.path("../../../data/rna/pseudobulk/outputs", gene_file))

  # Subset metadata to only relevant columns
  metadata_deseq2 <- metadata_sub %>%
    distinct(subject.subjectGuid, sample.sampleKitGuid, subject.biologicalSex, label.visitDetails)

  # Run DESeq2
  failed_log <- list()
  res_list <- lapply(celltypes, function(celltype) {
    tryCatch(
      {
        # Subset columns for this celltype from each df
        celltype_list <- lapply(df_list, function(df) {
          df[, grep(celltype, names(df), fixed = TRUE), drop = FALSE]
        })

        # Combine and normalize sample IDs
        exp_matrix <- do.call(cbind, celltype_list)
        colnames(exp_matrix) <- sub(":.*", "", colnames(exp_matrix))

        if (any(duplicated(colnames(exp_matrix)))) {
          print(paste("Removing", sum(duplicated(colnames(exp_matrix))), "duplicate columns for celltype:", celltype))
          # Take only the first occurrence of each sample
          exp_matrix <- exp_matrix[, !duplicated(colnames(exp_matrix)), drop = FALSE]
        }

        # Align metadata rownames to sample IDs
        rownames(metadata_deseq2) <- metadata_deseq2$sample.sampleKitGuid

        # Gene list for this celltype
        filtered_genes <- filtered_gene_set %>%
          filter(aifi_plot_l3 == celltype) %>%
          pull(gene)

        # Run DESeq2
        res <- deseq2_analysis(
          exp_matrix,
          meta_data = metadata_deseq2,
          filtered_gene_set = filtered_genes,
          formula = ~ subject.subjectGuid + label.visitDetails,
          comparisons = list(c("label.visitDetails", timepoint_1, timepoint_2)),
          celltype = celltype
        )

        # Post-process
        res <- as.data.frame(res)
        res$Qvalue <- qvalue::qvalue(res$pvalue)$qvalues
        return(res)
      },
      error = function(e) {
        failed_log <<- append(failed_log, list(data.frame(
          timestamp = Sys.time(),
          celltype  = celltype,
          error     = as.character(e)
        )))
        return(NULL)
      }
    )
  })

  # Save failed cell types log if any failures occurred
  if (length(failed_log) > 0) {
    failed_df <- do.call(rbind, failed_log)
    failed_outfile <- paste0(
      "../../../data/rna/pseudobulk/results/deseq2_results/failed_celltypes_pbmc_",
      timepoint_1, "_vs_", timepoint_2, ".csv"
    )
    write.csv(failed_df, file = failed_outfile, row.names = FALSE)
  }

  # Combine results and write to file
  res <- do.call(rbind, res_list[!sapply(res_list, is.null)])
  write.csv(res, file = file.path("../../../data/rna/pseudobulk/results/deseq2_results", out_file), row.names = FALSE)
}

[1] "Total reading time: 4.94499999999999 seconds"
[1] "The length of the list matches the length of the input path."


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

[1] "Removing 24 duplicate columns for celltype: Treg CD4 Mem"


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

[1] "Total reading time: 2.90299999999996 seconds"
[1] "The length of the list matches the length of the input path."


...

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

[1] "Total reading time: 1.24900000000002 seconds"
[1] "The length of the list matches the length of the input path."


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

[1] "Removing 12 duplicate columns for celltype: Treg CD4 Mem"


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting